[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordADH2026PythonBootcamp/blob/main/notebooks/05_L4_lists_and_loops.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util, subprocess, sys

if importlib.util.find_spec("adh_python") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/BradSegal/OxfordADH2026PythonBootcamp.git"],
        check=True,
    )
print("Course helpers ready.")

# Level 4: Lists and loops

A single observation tells you little, the trend tells you more. A **list**
keeps many values in order (like a day of respiratory rates) and a
**loop** lets you do the same thing to every value in it without writing a
line for each one. Together they let you summarise a whole observation
chart in a few lines.

**By the end of this level you can:**
- pick out readings from a list by position, including the latest;
- summarise a list with `len`, `sum`, `min`, `max` and `round`;
- write a loop that adds up, counts, finds the first match or builds a new
  list; and
- fix the two most common loop bugs.

**How the problems work.** As before, write your answer where you see `...`
and then run the cell to check it.

In [ ]:
from adh_python import clinical
from adh_python.checks import level4 as check
from adh_python.data import sample_patient

## A quick recap

In Level 3 you made rules with `if`, `elif`, and `else`. In Level 2, we saw
that we could use `.split("/")` to get a list of pieces of text.

In this level we are going to work with lists of readings.

The third import above gives you `sample_patient()`: Jordan Nguyen from the
basics notebook, admitted with community-acquired pneumonia. The cell below
takes one list each of the different types of observations from their chart.
Each set of square brackets selects a single entry by name (this is
explained in the Level 5 section). For now, just know that each line
produces a single list.

In [ ]:
observations = sample_patient()["observations"]
times = observations["time"]
respiration_rates = observations["respiration_rate"]
spo2_readings = observations["spo2"]
on_oxygen = observations["on_oxygen"]
pulses = observations["pulse"]
temperatures = observations["temperature"]
pain_scores = observations["pain_score"]

print(times)
print(respiration_rates)
print(temperatures)

Observations were taken in four hour intervals starting with the oldest. The
first and last observation are both at 06:00 but are a day apart.

## Lists and positions

A list is written using square brackets with each item separated by a comma.
A list preserves the order of elements and has a length that can be accessed
via the `len()` function.

Each item has a **position**, also called an index. Python counts positions
**from 0**, so the first item is at position 0, second at position 1, etc.
Negative positions are counted from the end, so `[-1]` is the last item (in
the case of an observation list, this is the latest reading).

In [ ]:
print(len(respiration_rates))
print(respiration_rates[0])  # the first (oldest) reading
print(respiration_rates[1])  # the second reading
print(respiration_rates[-1])  # the last (latest) reading
print(times[-1])

Asking for a position that does not exist, such as `respiration_rates[7]`
in a list of seven, stops with `IndexError: list index out of range`. The
last position is always one less than the length.

## Slices: part of a list

A **slice** is a way of getting a range of elements as a new list. This is
done by using `readings[start:stop]` which returns a new list from the
element at position `start` to the element **just before** position `stop`.
If you want to start at the beginning or go to the end, you may omit the
start or end value respectively.

- `readings[:3]` is the first three items;
- `readings[-3:]` is the last three items.

In [ ]:
print(respiration_rates[:3])
print(respiration_rates[-3:])

## Problem 4.1: Predict

Look at the `temperatures` list printed above. **Before you run anything**,
predict exactly what this will print:

```python
print(temperatures[1:3])
```

Store your prediction as a list, in square brackets, in `prediction`. Run
the check, then run the line itself in the cell after.

In [ ]:
prediction = ...
check.predict_slice(temperatures, prediction)

In [ ]:
print(temperatures[1:3])

## Summarising a list

Python has built-in functions that work on a whole list of numbers:
`sum()` adds them up, `min()` and `max()` give the smallest and largest,
and `len()` counts them. There is no built-in mean, but the mean is the sum
divided by the count. `round(value, 1)` rounds to one decimal place.

In [ ]:
mean_rr = sum(respiration_rates) / len(respiration_rates)
print(f"Mean respiratory rate: {round(mean_rr, 1)}")
print(f"Highest: {max(respiration_rates)}, lowest: {min(respiration_rates)}")

## Problem 4.2: Modify

The mean over the whole chart is hiding how much better Jordan is doing now.
This code finds the mean respiratory rate of every single reading. Alter the
**first line only** to make `recent_rates` only the 3 most recent readings
and take the mean of that instead.

In [ ]:
recent_rates = respiration_rates
mean_rr = sum(recent_rates) / len(recent_rates)
print(f"Readings used: {recent_rates}, mean {mean_rr:.1f}")
check.mean_recent(respiration_rates, mean_rr)

## Loops: doing something to every item

A `for` loop iterates through a list of items and performs the same set of
indented lines once for each item in the list. The variable name following
the `for` word is updated to be the next item in the list with each cycle.

```python
for temperature in temperatures:
    ...   # these indented lines run once per reading
```

The NEWS2 rules you wrote in Level 3 are also available pre-built and tested
within the course's `clinical` module that is imported at the top of this
notebook. For example, using `clinical.score_temperature(37.6)` provides the
NEWS2 temperature score for 37.6 °C. The loop below scores every temperature
on the chart:

In [ ]:
for temperature in temperatures:
    score = clinical.score_temperature(temperature)
    print(f"{temperature} °C scores {score}")

## Adding up as you go

Many loops follow the same three step pattern called an **accumulator**:

1. **before** the loop, create a variable to keep track of the running
   result;
2. **inside** the loop update it with each item;
3. **After** the loop, use the result.

This is the running fluid balance from Level 1 done by a loop. Here it is
summing the pain scores to determine the average.

In [ ]:
total = 0
for pain in pain_scores:
    total += pain
print(f"Total {total}, mean pain score {total / len(pain_scores):.1f}")

## Counting with a condition

Put an `if` inside the loop and you can count only the items that meet a
condition. The count starts at 0 before the loop and goes up by 1 only
when the condition is `True`. Here it counts the rounds on supplemental
oxygen; each item in `on_oxygen` is `True` or `False`.

In [ ]:
rounds_on_oxygen = 0
for oxygen in on_oxygen:
    if oxygen:
        rounds_on_oxygen += 1
print(f"Rounds on oxygen: {rounds_on_oxygen} of {len(on_oxygen)}")

## Problem 4.3: Make

How many observation rounds had an SpO2 that scored on NEWS2 (Scale 1)?
Count the readings in `spo2_readings` for which
`clinical.score_spo2_scale1(reading)` is more than 0.

Replace the `...` with what `count` should start as and then write the loop
below that.

In [ ]:
count = ...
print(f"Rounds with a scoring SpO2: {count}")
check.count_scoring_spo2(spo2_readings, count)

## Looping over positions with range

Sometimes you need the position as well as the value, for example if you
wanted to look up the time a particular reading occurred from the `times`
list. `range(len(times))` provides the positions 0, 1, 2, etc. up to but not
including the length (same rule as a slice). We know that each list has a
single item per round and so we can use the same position `i` to index
matching items from each.

In [ ]:
for i in range(len(times)):
    print(times[i], respiration_rates[i])

## Finding the first match

To find the **first** reading meeting a condition, you need to loop until
you find one, store it, then stop. Two new words we can use here are:

- `None` is Python's value for "nothing yet". Start with it, so that if
  nothing matches you can tell.
- `break` stops the loop straight away, skipping the remaining items.

This finds the first round where Jordan was off oxygen:

In [ ]:
first_on_air = None
for i in range(len(times)):
    if not on_oxygen[i]:
        first_on_air = times[i]
        break
print(f"First round on air: {first_on_air}")

## Problem 4.4: Make

What time was Jordan first apyrexial? Take this to mean the first round
where the temperature scored 0 on NEWS2, i.e.,
`clinical.score_temperature(temperatures[i]) == 0`.

This is a working definition for the exercise, not a clinical one. 38.0 °C
scores 0 on NEWS2 but is often treated as a fever locally. At the other end,
36.0 °C or below scores on NEWS2, and it would not be called apyrexial in
the usual sense either: it is a different abnormality, not a resolved
fever.

Store the **time** of that round as text in the same manner as the items in
`times` in `first_time`. Initialise this with `first_time = None`.

In [ ]:
first_time = ...
print(f"First apyrexial round: {first_time}")
check.first_apyrexial(temperatures, times, first_time)

## Building a new list

To build a list, start with an empty list `[]` before the loop and add to
the list inside the loop with `.append()`. Unlike the string methods in
Level 2, `.append()` modifies the list itself and so you do not store the
result.

In [ ]:
respiration_scores = []
for rate in respiration_rates:
    respiration_scores.append(clinical.score_respiration_rate(rate))
print(respiration_scores)

## Problem 4.5: Make

The loan thermometer from Level 1 is in Fahrenheit. Convert all the values
in `readings_f` to Celsius and round it to a single decimal place. Save the
results as a new list called `readings_c` in the same order.

The formula is the same as you did in Level 1, subtract 32 and then multiply
by 5/9.

In [ ]:
readings_f = [101.3, 100.4, 99.5, 98.2, 97.0]
readings_c = ...
print(readings_c)
check.convert_temperatures(readings_f, readings_c)

## Problem 4.6: Fix the bug

This code should find the mean pulse of Jordan over the entire chart. This
runs without an error but prints out a mean pulse of 12.0 which is clearly
not a plausible value for a list starting at 112. There are two bugs that
have snuck in, find them and fix them.

In [ ]:
for i in range(1, len(pulses)):
    total = 0
    total += pulses[i]
mean = total / len(pulses)
print(f"Mean pulse: {mean:.1f}")
check.mean_pulse(pulses, mean)

<details>
<summary><b>Why did that happen?</b> Open this after your own attempt.</summary>

`total = 0` was inside the loop and so the total was being reset every round
and only the very last item was being kept. We need to create this once
prior to the loop. The `range(1, len(pulses))` also skipped the 0th position
so was ignoring the first pulse. The much simpler `for pulse in pulses:`
does not have this issue and if you wanted to make it even simpler you could
use `sum(pulses)` which does this in a single line. The for loop just
demonstrates how the `sum` function is working.

</details>

## Before you move on

- Positions start from 0, with `[-1]` being the latest reading.
- A slice `[start:stop]` stops just before `stop`.
- An accumulator is made once **before** the loop, updated within the loop,
  and used after the loop.
- To find the first match, store it and `break`. Without `break` you get
  the last match.
- `.append()` adds to a list inside a loop; start from `[]` before it.

**Next:** Level 5, dictionaries: keeping a patient's details together under
names.